# 03 Error analysis (validation only)

Draft. The best run is chosen by **validation** Macro F1; test is not looked at here. Built by `scripts/error_analysis.py notebook`. Only English glosses (Gemini), labels and ids are shown; original comment text stays out of the notebook.

In [1]:
import json, pandas as pd
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
import sys; sys.path.insert(0, str(ROOT / "src"))
from terralens.evaluate import confusion_df, per_class_report
sample = pd.read_csv(ROOT / "data/interim/error_sample.csv", dtype=str, keep_default_na=False)
RUN = sample["run"].iloc[0]
res = json.loads((ROOT / "results" / f"{RUN}.json").read_text(encoding="utf-8"))
pred = pd.read_csv(ROOT / "results/predictions" / f"{RUN}_val.csv", dtype=str, keep_default_na=False)
print("Best run by val Macro F1:", RUN, "| val Macro F1", res["val"]["macro_f1"], "| val Against F1", res["val"]["against_f1"])

Best run by val Macro F1: e4_qlora | val Macro F1 0.6866 | val Against F1 0.45


## Confusion matrix and per-class scores (val)

In [2]:
print(confusion_df(pred['gold'], pred['pred']))
print(per_class_report(pred['gold'], pred['pred']).round(3))

pred     Favour  Against  None
gold                          
Favour       57        5    12
Against       2        9     8
None         34        7   388
         precision  recall     f1  support
Favour       0.613   0.770  0.683       74
Against      0.429   0.474  0.450       19
None         0.951   0.904  0.927      429


## Macro F1 per language, script and annotator tag (val)

In [3]:
for k in ('by_lang', 'by_script', 'by_tag'):
    print(k); display(pd.DataFrame(res['val'][k]).T)

by_lang


,n,macro_f1,against_f1
bn,165.0,0.6178,0.6667
hi,357.0,0.6571,0.3571


by_script


,n,macro_f1,against_f1
bengali,130.0,0.6143,0.6667
devanagari,150.0,0.6633,0.3529
latin,226.0,0.6493,0.3636
mixed,16.0,0.5877,0.0000


by_tag


,n,macro_f1,against_f1,accuracy
sarcasm,19.0,0.4660,0.6154,0.6842
implicit,44.0,0.3511,0.0000,0.7500
code_mixed,63.0,0.7550,0.4444,0.8889
romanised,269.0,0.6436,0.3333,0.8625
question,28.0,0.3270,0.0000,0.9286


## 40 misclassified validation comments

Sampled with seed 42, spread over gold -> prediction pairs. `category` is a hand-assigned error type (see `reports/error_categories.csv`).

In [4]:
gl = {}
p = ROOT / 'data/interim/error_glosses.jsonl'
for line in p.read_text(encoding='utf-8').splitlines():
    r = json.loads(line); gl[r['id']] = r['gloss']
cat = pd.read_csv(ROOT / 'reports/error_categories.csv', dtype=str) if (ROOT / 'reports/error_categories.csv').exists() else pd.DataFrame(columns=['id','category'])
view = sample.assign(gloss=sample['id'].map(gl)).merge(cat, on='id', how='left')
pd.set_option('display.max_colwidth', 160)
display(view[['label', 'pred', 'lang', 'script', 'tags', 'category', 'gloss']].rename(columns={'label': 'gold'}))

,gold,pred,lang,script,tags,category,gloss
0,None,Against,hi,devanagari,[],"blame, politics or conspiracy","What is said is correct. Most scientists were men. If they hadn't made inventions, the earth would be safe today."
1,None,Against,bn,bengali,"[""romanised""]",weather facts without stance,"This is also true. But brother, you cannot dismiss what meteorologists say. From what I saw, this year there has been 45% rainfall. There is a 55% deficit. ..."
2,Favour,None,hi,devanagari,"[""implicit""]",needs video context,"🌸 🌸 She is saying that men are ruling the world.. giving orders, waging wars, setting fires, conducting nuclear tests, spreading pollution 🌸 collision"
3,None,Favour,hi,latin,"[""romanised""]","topic, not target (pollution, population, trees)","No no sir, POISON is being sold in the name of milk."
4,Against,None,bn,bengali,"[""sarcasm""]",sarcasm,"I heard the news that this year 2026 too would have the worst heat, and heard about a strong El Nino 😅 but where is it 😅"
5,Favour,None,hi,devanagari,"[""implicit""]",implicit stance,"No sir, in 1960 the population in villages was low. The number of animals was higher than humans. Factories and vehicles were almost non-existent. There wer..."
6,None,Favour,bn,bengali,[],weather facts without stance,This is 100% true. A Super El Nino might come in 2027. But its impact won't be the same everywhere. It will have different effects in different places.
7,None,Favour,hi,latin,"[""romanised""]",generic agreement or praise,"The same thing is happening with me too, but let it happen, we will keep doing our work"
8,None,Favour,hi,latin,"[""romanised""]",needs video context,"You said it absolutely right.. these people have gone after animals, humans have made their eating, drinking, and living all miserable.. they keep saying do..."
9,Favour,Against,hi,devanagari,"[""code_mixed"", ""romanised""]",cause attribution read as denial,"We are not paying attention to where the problem is, our attention is somewhere else. Like the real causes of climate change are increasing population, carb..."


## Error categories

In [5]:
display(view.groupby('category').agg(n=('id', 'size'), pairs=('pair', lambda s: s.value_counts().to_dict())).sort_values('n', ascending=False))

,n,pairs
category,,
"blame, politics or conspiracy",6,"{'None->Favour': 2, 'Against->None': 2, 'None->Against': 1, 'Against->Favour': 1}"
generic agreement or praise,5,{'None->Favour': 5}
"topic, not target (pollution, population, trees)",5,{'None->Favour': 5}
sarcasm,4,"{'Against->None': 2, 'Against->Favour': 1, 'None->Favour': 1}"
implicit stance,4,{'Favour->None': 4}
needs video context,4,"{'Favour->None': 2, 'None->Favour': 2}"
cause attribution read as denial,3,{'Favour->Against': 3}
weather facts without stance,3,"{'None->Against': 2, 'None->Favour': 1}"
request or question about the topic,3,{'None->Favour': 3}
